In [1]:
import duckdb

In [2]:
conn = duckdb.connect()

In [3]:
conn.execute("""
    CREATE OR REPLACE SECRET s3_secret (
        TYPE s3,
        PROVIDER config,
        KEY_ID 'ozone-access-key',
        SECRET 'ozone-secret-key',
        ENDPOINT 'ozone-s3g:9878',
        URL_STYLE 'path',
        USE_SSL false,
        REGION 'us-east-1'
    );
""")

conn.execute("""
    ATTACH 'warehouse' AS iceberg_catalog (
        TYPE iceberg,
        AUTHORIZATION_TYPE 'NONE',
        ENDPOINT 'http://nessie-catalog:19120/iceberg/',
        ACCESS_DELEGATION_MODE 'NONE'
    );
""")

conn.execute("""
    SET s3_endpoint='ozone-s3g:9878';
    SET s3_access_key_id='ozone-access-key';
    SET s3_secret_access_key='ozone-secret-key';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
    SET s3_region='us-east-1';
""")

In [4]:
res = conn.execute("SHOW ALL TABLES;").fetchdf()
print(res)

          database                            schema                name  \
0  iceberg_catalog      cryptocurrencies_project_raw                 btc   
1  iceberg_catalog  cryptocurrencies_project_raw_stg                 btc   
2  iceberg_catalog                   test_iceberg_db  test_iceberg_table   
3  iceberg_catalog                 test_iceberg_db_2  test_iceberg_table   

  column_names column_types  temporary  
0         [__]    [UNKNOWN]      False  
1         [__]    [UNKNOWN]      False  
2         [__]    [UNKNOWN]      False  
3         [__]    [UNKNOWN]      False  


In [5]:
conn.execute("INSERT INTO iceberg_catalog.cryptocurrencies_project_raw_stg.btc VALUES ('2025-01-01', 1, 2, 3, 4, 5, NULL);")

In [4]:
res = conn.execute("SELECT * FROM iceberg_catalog.test_iceberg_db_2.test_iceberg_table;").fetchdf()
res.head(10)

,id,descr,report_dt
0,1115,spark job test,2025-12-31
1,1115,spark job test,2025-12-31
2,1,tgete,2025-02-01
3,3,vdsv,2025-06-01
4,1,tgete,2025-02-01
5,3,vdsv,2025-06-01
